# Lesson 17: Team and Batch Processing

The last 2 lessons built the individual agents: Content Writer (Lesson 15), Image Finder and AIO Analyzer (Lesson 16). Now **connect them** into a coordinated team:

```
Your request --> [Team Leader] --> delegates to --> [Content Writer / Image Finder / AIO Analyzer]
```

This is the **actual product architecture**. We import directly from the source.

What's different from running agents individually:
- **Team Leader** — reads your request and delegates to the right member
- **Parallel processing** — batch creates multiple articles simultaneously
- **Task mode** — the team can run multiple tasks concurrently

## Agno Team

An Agno **Team** is a group of agents managed by a leader:

```
Team Leader (Claude Sonnet) -- reads your request, decides who handles it
  |
  +-- Content Writer -- researches and writes articles
  +-- Image Finder -- finds and inserts images (optional)
  +-- AIO Analyzer -- analyzes Google AI Overviews
```

Key concept: **TeamMode.tasks** lets the leader run multiple tasks in parallel.

When you say "Create articles about Topic A, Topic B, Topic C", the leader creates 3 parallel tasks instead of processing them one by one.

## Error Handling: try / except

What happens when something goes wrong? An API call fails, a search returns nothing, or the network drops.

Python uses **try/except** to handle errors gracefully:

```python
try:
    # Try running this code
    result = agent.run("...")
except Exception as e:
    # If it fails, do this instead
    print(f"Something went wrong: {e}")
```

- Code inside `try` runs normally
- If any line inside `try` fails, Python **jumps** to `except` immediately
- The variable `e` contains the error message
- Without try/except, the whole program crashes

**One failing article doesn't crash the whole batch** — other articles continue processing normally.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## The Team Definition

Let's look at the actual team code from `output/backend/agents/team.py`:

In [ ]:
# Show the actual team definition
team_path = os.path.abspath("../../output/backend/agents/team.py")
with open(team_path, "r", encoding="utf-8") as f:
    print(f.read())

Key points from the team definition:

- `id="seo-workspace"` — identifies the team in the API
- `mode=TeamMode.tasks` — enables parallel task execution
- `members=[content_writer, aio_analyzer]` + optionally `image_finder`
- **Image Finder is conditional** — only added if DataForSEO is configured
- `db=SqliteDb(...)` — chat history persisted in SQLite
- `store_member_responses=True` — keeps member responses for context

## Single Article Creation

First, let's create one article through the team:

> **Cost:** ~$0.20-0.50 per article (team leader + content writer Sonnet calls). Takes 1-2 minutes.

In [ ]:
from agents.team import team

topic = "10 Simple SEO Tips for Beginners"
print(f"Creating article: {topic}\n")

response = team.run(f"Write an SEO article about: {topic}")
print(response.content[:2000])
if len(response.content) > 2000:
    print("\n... (truncated)")

In [ ]:
# Check what was saved
import json
from tools.storage import list_all_articles, get_article_content

result = list_all_articles()
articles = json.loads(result)
print(f"Articles in storage: {len(articles)}")
for a in articles:
    print(f"  {a['id']}: {a['topic']} ({a['word_count']} words, {a['status']})")

## Reading Files in Python

Articles are saved as `.md` files in the `content/` directory:

```python
with open("filename.md", "r", encoding="utf-8") as f:
    content = f.read()
```

Breaking this down:
- `open(path, "r")` — opens a file for **r**eading (`"w"` would be for **w**riting)
- `encoding="utf-8"` — handles special characters correctly
- `with ... as f:` — automatically closes the file when done
- `f.read()` — reads the entire file contents into a string

In [ ]:
# Read the saved article file
if articles:
    last_article = articles[-1]
    content_json = get_article_content(last_article['id'])
    content = json.loads(content_json)
    print(f"Article: {content['topic']}\n")
    print(content['article_markdown'][:2000])
    if len(content.get('article_markdown', '')) > 2000:
        print("\n... (truncated for display)")

## Batch Processing — Multiple Articles in Parallel

The real power of `TeamMode.tasks`: creating **multiple articles at once**.

Instead of creating articles one by one (sequential), the team runs them in parallel. With 3 articles, that means ~3x faster.

> **Cost:** ~$0.60-1.50 for 3 articles. Takes 2-4 minutes total (vs 6-12 minutes sequential).

In [ ]:
import time

topics = [
    "What is Technical SEO and Why It Matters",
    "How to Write Meta Descriptions That Get Clicks",
    "Internal Linking Strategy for Better Rankings",
]

print(f"Creating {len(topics)} articles in batch...\n")
start = time.time()

# Send all topics in one prompt — the team leader creates parallel tasks
prompt = "Write SEO articles about each of these topics:\n"
for i, t in enumerate(topics, 1):
    prompt += f"{i}. {t}\n"

response = team.run(prompt)

elapsed = time.time() - start
print(f"\nDone in {elapsed:.0f} seconds")
print(response.content[:3000])

In [ ]:
# See the results
result = list_all_articles()
articles = json.loads(result)
print(f"Total articles in storage: {len(articles)}\n")

total_words = 0
for a in articles:
    wc = a.get('word_count', 0) or 0
    total_words += wc
    print(f"  {a['id']}: {a['topic']}")
    print(f"    {wc} words, status: {a['status']}")

print(f"\nTotal words across all articles: {total_words:,}")

## Exercise

Using the `articles` list from above, check the article quality:

1. Is the word count above 1000 for each article? Print a pass/fail for each.
2. What status are the articles in? (should be `"review"` if successful)
3. Count how many articles are in each status.

In [ ]:
# Exercise: Fill in the blanks to check article quality

# 1. Check word count for each article
for a in articles:
    wc = a.get(___, 0) or 0
    status = "PASS" if wc > ___ else "FAIL"
    print(f"{status}: {a['topic']} ({wc} words)")

# 2. Print each article's status
print("\nArticle statuses:")
for a in articles:
    print(f"  {a['id']}: {a[___]}")

# 3. Count articles by status
status_counts = {}
for a in articles:
    s = a["status"]
    status_counts[s] = status_counts.get(s, 0) + ___
print(f"\nStatus counts: {status_counts}")

<details>
<summary>Click to reveal answer</summary>

```python
# 1. Check word count for each article
for a in articles:
    wc = a.get("word_count", 0) or 0
    status = "PASS" if wc > 1000 else "FAIL"
    print(f"{status}: {a['topic']} ({wc} words)")

# 2. Print each article's status
print("\nArticle statuses:")
for a in articles:
    print(f"  {a['id']}: {a['status']}")

# 3. Count articles by status
status_counts = {}
for a in articles:
    s = a["status"]
    status_counts[s] = status_counts.get(s, 0) + 1
print(f"\nStatus counts: {status_counts}")
```
</details>

## Done

A complete **AI-powered SEO team** with batch processing:

- **3 specialized agents**, each using Claude Sonnet with focused tools
- **Team Leader** delegates requests to the right member
- **Batch processing** — multiple articles created in parallel
- **Local storage** — articles saved as `.md` files with metadata in JSON

This code runs the same way in the real product. When you run:
```bash
python output/backend/serve.py
```
And send a prompt through the web interface, it uses the same team we tested here.

### Next: Module 5

The next module completes the product with:
- Local storage layer — how articles are persisted
- How everything connects — the full architecture
- Web interface — the browser-based chat
- Extending the product — adding new capabilities with vibecoding